In [ ]:
# @title Run Processing Interface
print("🔄 Initializing system interface & dependencies...")
import os
import sys
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import folium
import branca.colormap as b_cmp
from branca.element import Figure
from rasterio.warp import transform_bounds, calculate_default_transform, reproject
from rasterio.enums import Resampling
from IPython.display import clear_output, display, HTML
import requests
import subprocess
from google.colab import output as colab_output

# 2. Silently verify or install Google SDK layers in the background
try:
    from google.cloud import batch_v1
    from google.cloud import storage
    from google.cloud import bigquery
except ImportError:
    !pip install -q google-cloud-batch google-cloud-storage google-cloud-bigquery folium branca > /dev/null 2>&1
    from google.cloud import batch_v1
    from google.cloud import bigquery
    from google.cloud import storage

# 3. Suppress underlying third-party library warning noise
import warnings
warnings.filterwarnings("ignore")

# 4. Wipe cell clean to eliminate console walls and the red warning box
clear_output()

# 5. Import core streamlined runtime modules
import time
import uuid
import shutil
import geopandas as gpd
import rasterio
import ipywidgets as widgets
from datetime import date, timedelta, datetime
from rasterio.mask import mask
from google.colab import files
from google.colab import auth

# ==================================================
# 🚨 NATIVE COLAB MAP BRIDGE SETUP
# ==================================================

# Global variable to hold the polygon drawn on the map
global_drawn_polygon = None
current_job_id_tab1 = None
system_is_rendering_map = False  # 🚦 NEW: Our traffic light flag

def receive_polygon(geojson_str):
    """Callback function triggered by Javascript when a polygon is drawn."""
    global global_drawn_polygon
    global_drawn_polygon = json.loads(geojson_str)

# Register the callback so Javascript can find it
colab_output.register_callback('notebook.receive_polygon', receive_polygon)

# Raw HTML/JS code for the map
html_map_code = """
<!DOCTYPE html>
<html>
<head>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet.draw/1.0.4/leaflet.draw.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet.draw/1.0.4/leaflet.draw.js"></script>
    <style>
        #leaflet-map { height: 400px; width: 100%; border: 2px solid var(--jp-border-color1, #888); border-radius: 5px; z-index: 1;}
    </style>
</head>
<body>
    <div id="leaflet-map"></div>
    <div id="draw-status" style="padding-top: 10px; font-family: sans-serif; color: #EE6723; font-weight: bold;">
        Draw a polygon on the map. It will be used to clip the data.
    </div>
    <script>
        var map = L.map('leaflet-map').setView([-28.0, 133.0], 4);
        L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png', {
            maxZoom: 18, attribution: '© OpenStreetMap'
        }).addTo(map);

        var drawnItems = new L.FeatureGroup();
        map.addLayer(drawnItems);

        var drawControl = new L.Control.Draw({
            draw: {
                polygon: { shapeOptions: { color: '#EE6723', fillColor: '#EE6723', fillOpacity: 0.4 } },
                polyline: false, rectangle: false, circle: false, marker: false, circlemarker: false
            },
            edit: { featureGroup: drawnItems }
        });
        map.addControl(drawControl);

        map.on(L.Draw.Event.CREATED, function (e) {
            var layer = e.layer;
            drawnItems.clearLayers();
            drawnItems.addLayer(layer);
            var geojsonData = layer.toGeoJSON();

            // Send data to Python and update UI text
            google.colab.kernel.invokeFunction('notebook.receive_polygon', [JSON.stringify(geojsonData)], {});
            document.getElementById('draw-status').innerHTML = "✅ Polygon captured! Ready for download.";
        });
    </script>
</body>
</html>
"""

# Clear structural credential overrides if present
if "GOOGLE_APPLICATION_CREDENTIALS" in os.environ:
    del os.environ["GOOGLE_APPLICATION_CREDENTIALS"]

directories_to_clear = [
    "/tmp/paleo_downloads",
    "/tmp/paleo_preview"
]

# Wipe old folders
for dir_path in directories_to_clear:
    if os.path.exists(dir_path):
        shutil.rmtree(dir_path)

# Delete the old zip file if it exists
if os.path.exists("/content/Paleo_Processed_Data.zip"):
    os.remove("/content/Paleo_Processed_Data.zip")

# 1. Check if we are in a Colab environment at all
is_colab = "google.colab" in sys.modules
is_enterprise = "VERTEX_PRODUCT" in os.environ or "IS_COLAB_ENTERPRISE" in os.environ

user_email = "localuser"

if is_colab and is_enterprise:
    result = subprocess.run(['gcloud', 'config', 'get-value', 'account'], capture_output=True, text=True)
    user_email = result.stdout.strip()
elif is_colab:
    try:
        auth.authenticate_user()
        gcloud_token = !gcloud auth print-access-token
        url = f"https://www.googleapis.com/oauth2/v3/tokeninfo?access_token={gcloud_token[0]}"
        token_data = requests.get(url).json()
        user_email = token_data.get('email', 'localuser')
    except Exception:
        pass

# ==================================================
# 1. BATCH JOB CONFIGURATION & SUBMISSION
# ==================================================

def create_and_submit_batch_job(envs, variable, model, start_date, end_date, dataset, use_spot=False):
    global user_email
    PROJECT_ID = "paleo-interpolation"
    REGION = "australia-southeast2"
    CONTAINER_IMAGE = "australia-southeast2-docker.pkg.dev/paleo-interpolation/paleo-repo/paleo-geotiff-v2:latest"

    client = batch_v1.BatchServiceClient()

    runnable = batch_v1.Runnable()
    runnable.container = batch_v1.Runnable.Container()
    runnable.container.image_uri = CONTAINER_IMAGE

    task_spec = batch_v1.TaskSpec()
    task_spec.max_retry_count = 10
    task_spec.runnables = [runnable]
    task_spec.compute_resource = batch_v1.ComputeResource()
    task_spec.compute_resource.cpu_milli = 16000
    task_spec.compute_resource.memory_mib = 60000
    task_spec.max_run_duration = "14400s"

    task_group = batch_v1.TaskGroup()
    task_group.task_spec = task_spec
    task_group.task_environments = envs
    task_group.task_count = len(envs)
    task_group.parallelism = 46

    allocation_policy = batch_v1.AllocationPolicy()
    instance_policy = batch_v1.AllocationPolicy.InstancePolicyOrTemplate()
    instance_policy.policy = batch_v1.AllocationPolicy.InstancePolicy()
    instance_policy.policy.machine_type = "n2-standard-16"

    if use_spot:
        instance_policy.policy.provisioning_model = batch_v1.AllocationPolicy.ProvisioningModel.SPOT
    else:
        instance_policy.policy.provisioning_model = batch_v1.AllocationPolicy.ProvisioningModel.STANDARD

    allocation_policy.instances = [instance_policy]

    logs_policy = batch_v1.LogsPolicy()
    logs_policy.destination = batch_v1.LogsPolicy.Destination.CLOUD_LOGGING

    job = batch_v1.Job()
    job.task_groups = [task_group]
    job.allocation_policy = allocation_policy
    job.logs_policy = logs_policy

    raw_name = user_email.split('@')[0].split('.')[0].lower()
    safe_user = re.sub(r'[^a-z0-9]', '', raw_name)[:4]
    if not safe_user:
        safe_user = "user"

    var_lower = variable.lower()
    if 'rain' in var_lower: safe_var = 'r'
    elif 'mwet' in var_lower: safe_var = 'mw'
    elif 'mlake' in var_lower: safe_var = 'ml'
    elif 'fao56' in var_lower or 'foa56' in var_lower: safe_var = 'f'
    else: safe_var = re.sub(r'[^a-z0-9]', '', var_lower)[:2]

    if 'Regression' in model: safe_model = 'rksvm'
    elif 'IDW' in model: safe_model = 'idw'
    elif 'Spline' in model: safe_model = 'spl'
    else: safe_model = re.sub(r'[^a-z0-9]', '', model.lower())[:5]

    safe_start = start_date[:4] if start_date else "0000"
    safe_end = end_date[:4] if end_date else "9999"

    submit_time = datetime.now().strftime("%y%m-%H%M")

    job_id = f"p-{safe_user}-{safe_var}-{safe_model}-{safe_start}-{safe_end}-{submit_time}-{uuid.uuid4().hex[:4]}"

    request = batch_v1.CreateJobRequest(
        parent=f"projects/{PROJECT_ID}/locations/{REGION}",
        job_id=job_id,
        job=job,
    )

    try:
        response = client.create_job(request=request)
        return job_id
    except Exception as e:
        raise Exception(f"SDK Submission Failed: {str(e)}")

def generate_batch_environments(global_start, global_end, model, variable, user_email, dataset):
    start_year_str = global_start[:4]
    end_year_str = global_end[:4]

    yymmdd_hhmm = datetime.now().strftime("%y%m%d_%H%M")

    current_run_date = f"{user_email}_{start_year_str}_{end_year_str}_{yymmdd_hhmm}"

    start_year = int(start_year_str)
    end_year = int(end_year_str)
    first_decade = (start_year // 10) * 10
    last_decade = (end_year // 10) * 10

    batch_environments = []

    for decade_start_year in range(first_decade, last_decade + 10, 10):
        if decade_start_year == first_decade:
            task_start = global_start
        else:
            task_start = f"{decade_start_year:04d}-01-01"

        if decade_start_year == last_decade:
            task_end = global_end
        else:
            decade_end_year = decade_start_year + 9
            task_end = f"{decade_end_year:04d}-12-31"

        env = batch_v1.Environment(
            variables={
                "START_DATE": task_start,
                "END_DATE": task_end,
                "MODEL": model,
                "VARIABLE": variable,
                "CURRENT_RUN_DATE": current_run_date,
                "DATASET": dataset
            }
        )
        batch_environments.append(env)

    return batch_environments

# ==================================================
# 2. INTERFACE & UI ENVIRONMENT ASSEMBLY
# ==================================================

def interface():
    date_error_msg = widgets.HTML(value="", layout=widgets.Layout(width='100%', margin='-15px 0 10px 0'))
    comp_layout = widgets.Layout(overflow='hidden', width='100%')
    description_style = {'description_width': 'initial'}
    spacer = widgets.HTML(value="<div style='margin-bottom: 20px;'></div>")

    # --- Checkboxes Layout ---
    run_full_dataset = widgets.Checkbox(value=True, description='Process Specific Simulation Years', indent=False, layout=widgets.Layout(width='49%'))
    run_spot_vms = widgets.Checkbox(value=True, description='Use SPOT VMs (Cost-Saving)', indent=False, layout=widgets.Layout(width='49%'))
    checkbox_row_tab1 = widgets.HBox([run_full_dataset, run_spot_vms], layout=widgets.Layout(width='100%', margin='0 0 10px 0'))

    # --- Updated Year-Only Inputs ---
    start_date_core = widgets.Text(placeholder='YYYY (e.g., 0000)', description='Simulation Start Year', layout=comp_layout, style=description_style)
    end_date_core = widgets.Text(placeholder='YYYY (e.g., 9999)', description='Simulation End Year', layout=comp_layout, style=description_style)

    date_layout_split = widgets.Layout(overflow='hidden', width='49.5%', margin='0 0.25% 0 0.25%')
    start_date_final = widgets.VBox([start_date_core], layout=date_layout_split)
    end_date_final = widgets.VBox([end_date_core], layout=date_layout_split)

    date_spacer = widgets.HTML(value="<div style='margin-bottom: 20px;'></div>")
    date_row = widgets.HBox([start_date_final, end_date_final], layout=widgets.Layout(overflow='hidden', width='100%'))

    dropdown = widgets.Dropdown(options=['Rain', 'FAO56', 'Mwet'], value='Rain', description='Select Variable', layout=comp_layout, style=description_style)
    dropdown_final = widgets.VBox([dropdown], layout=comp_layout)

    dataset_dropdown = widgets.Dropdown(options=['Loading...'], description='Select Dataset', layout=comp_layout, style=description_style)
    dataset_dropdown_final = widgets.VBox([dataset_dropdown], layout=comp_layout)

    dropdown_model = widgets.Dropdown(options=['Regression Kriging: Support Vector Machine', 'IDW: Inverse Distance Weighting', 'Spline'], value='Regression Kriging: Support Vector Machine', description='Select Model', layout=comp_layout, style=description_style)
    dropdown_model_final = widgets.VBox([dropdown_model], layout=comp_layout)

    submit_btn = widgets.Button(description='Submit', layout=widgets.Layout(overflow='hidden', width='100%', height='50px'))
    submit_btn.style.button_color = '#EE6723'
    submit_btn.style.font_weight = 'bold'

    cancel_btn_1 = widgets.Button(description='Cancel Job', button_style='danger', layout=widgets.Layout(overflow='hidden', width='49%', height='50px', display='none', margin='0 0 0 2%'))
    cancel_btn_1.style.font_weight = 'bold'

    tab1_buttons = widgets.HBox([submit_btn, cancel_btn_1], layout=widgets.Layout(width='100%'))

    progress_bar = widgets.IntProgress(value=0, min=0, max=10, description='Progress:', bar_style='info', layout=widgets.Layout(overflow='hidden', width='100%', display='none'))
    output_display = widgets.Textarea(value="", description='Result', disabled=True, layout=widgets.Layout(overflow='hidden', width='100%', height='100px', display='none'), style=description_style)
    output_final = widgets.VBox([output_display], layout=comp_layout)

    def update_datasets(*args):
        current_var = dropdown.value
        dataset_dropdown.options = ['Loading from BigQuery...']

        try:
            # Using the hardcoded project ID
            bq_client = bigquery.Client(project="paleo-interpolation")

            # 1. Point explicitly to the Paleo_Data dataset
            dataset_ref = bq_client.dataset("Paleo_Data")

            # 2. List the TABLES inside that dataset, not the datasets themselves
            tables = list(bq_client.list_tables(dataset_ref))

            # 3. Filter tables to match "{variable}_Data%"
            prefix = f"{current_var}_Data"

            # 4. Format the output string exactly how you want it: project.dataset.table
            valid_tables = [f"paleo-interpolation.Paleo_Data.{t.table_id}" for t in tables if t.table_id.startswith(prefix)]

            if valid_tables:
                dataset_dropdown.options = valid_tables
            else:
                dataset_dropdown.options = ["No matching tables found"]

        except Exception as e:
            dataset_dropdown.options = [f"BigQuery Error: Check permissions"]

    dropdown.observe(update_datasets, names='value')
    update_datasets()

    def hide_dates(change):
        if change['new'] == False:
            date_row.layout.display = 'none'
            date_spacer.layout.display = 'none'
        else:
            date_row.layout.display = 'flex'
            date_spacer.layout.display = 'block'

    def submit_tasks():
        selected_var = dropdown.value
        selected_model = dropdown_model.value
        start_year_val = start_date_core.value.strip()
        end_year_val = end_date_core.value.strip()
        selected_spot = run_spot_vms.value
        selected_dataset = dataset_dropdown.value

        if selected_model == 'Regression Kriging: Support Vector Machine': selected_model = 'RK_SVM_Spherical'
        elif selected_model == 'IDW: Inverse Distance Weighting': selected_model = 'IDW'
        elif selected_model == 'Spline': selected_model = 'Spline'

        if start_year_val == "": start_Date = "0000-01-01"
        else: start_Date = f"{start_year_val}-01-01"

        if end_year_val == "": end_Date = "9999-12-31"
        else: end_Date = f"{end_year_val}-12-31"

        envs = generate_batch_environments(start_Date, end_Date, selected_model, selected_var, user_email, selected_dataset)
        try:
            job_id = create_and_submit_batch_job(envs, selected_var, selected_model, start_Date, end_Date, selected_dataset, use_spot=selected_spot)
            return job_id, len(envs)
        except Exception as e:
            return None, str(e)

    def check_job_once(job_id, total_tasks, client):
        """Checks GCP exactly once, updates the UI, and returns True if finished."""
        PROJECT_ID = "paleo-interpolation"
        REGION = "australia-southeast2"
        job_name = f"projects/{PROJECT_ID}/locations/{REGION}/jobs/{job_id}"

        try:
            job_data = client.get_job(name=job_name)
            state_obj = job_data.status.state
            state = state_obj.name if hasattr(state_obj, "name") else str(state_obj)

            state_map = {"1": "QUEUED", "2": "SCHEDULED", "3": "RUNNING", "4": "SUCCEEDED", "5": "FAILED"}
            state = state_map.get(state, state)

            if job_data.status.task_groups and len(job_data.status.task_groups) > 0:
                group_status = list(job_data.status.task_groups.values())[0]
                counts = group_status.counts

                succeeded = counts.get("SUCCEEDED", 0)
                failed = counts.get("FAILED", 0)
                running = counts.get("RUNNING", 0)
                pending = counts.get("PENDING", 0)
                assigned = counts.get("ASSIGNED", 0)

                spec_groups = job_data.task_groups
                true_total = spec_groups[0].task_count if len(spec_groups) > 0 else total_tasks
                active_sum = succeeded + failed + running + pending + assigned
                dormant = max(0, true_total - active_sum)

                prog_val = min(succeeded + failed, true_total)

                msg = (
                    f"Job ID: {job_id}\n"
                    f"Overall State: {state} | True Total: {true_total}\n"
                    f"Tasks -> Succeeded: {succeeded} | Failed: {failed} | Running: {running} | Pending: {pending} | Dormant: {dormant}"
                )
            else:
                msg = f"Job ID: {job_id}\nState: {state}\nAllocating infrastructure nodes..."
                prog_val = 0

            output_display.value = msg
            progress_bar.value = prog_val

            # If it's done, return True to tell the secret timer to stop
            if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
                output_display.value += f"\n\n✅ Process completed with final status: {state}"
                progress_bar.value = total_tasks
                return True

            return False # Still running

        except Exception as e:
            if "404" in str(e) or "Not Found" in str(e):
                return True # Job is gone, tell timer to stop
            else:
                output_display.value = f"Tracking status, retrying... (Connection note: {str(e)})"
                return False

    def js_check_status_callback():
        global current_job_id_tab1
        global system_is_rendering_map # 1. Bring in the flag

        # 2. TRAFFIC LIGHT: If Tab 3 is busy rendering a map, skip this 15s tick entirely!
        if system_is_rendering_map:
            return

        # 1. Look at the Cancel Button FIRST
        if cancel_btn_1.disabled:
            output_display.value += "\n\n🛑 Polling stopped. Job was cancelled."
            submit_btn.disabled = False
            submit_btn.layout.width = '100%'
            cancel_btn_1.layout.display = 'none'
            # Tell JS to kill the timer
            display(HTML("<script>if(window.jobPollInterval) clearInterval(window.jobPollInterval);</script>"))
            return

        if not current_job_id_tab1:
            return

        try:
            client = batch_v1.BatchServiceClient()
            # Trigger the single check
            is_finished = check_job_once(current_job_id_tab1, progress_bar.max, client)

            # If the job is done, clean up the UI and kill the timer
            if is_finished:
                submit_btn.disabled = False
                submit_btn.layout.width = '100%'
                cancel_btn_1.layout.display = 'none'
                display(HTML("<script>if(window.jobPollInterval) clearInterval(window.jobPollInterval);</script>"))

        except Exception as e:
            output_display.value += f"\n⚠️ Polling hiccup: {e}"

    # Register the callback so JS can access it!
    colab_output.register_callback('notebook.check_status', js_check_status_callback)

    def process_data_internal(button):
        output_display.value = "Communicating with Google Cloud... Please wait."
        progress_bar.value = 0
        progress_bar.layout.display = 'flex'
        output_display.layout.display = 'flex'
        submit_btn.disabled = True

        job_id, result = submit_tasks()

        if not job_id:
            output_display.value = f"❌ Failed to submit job:\n{result}"
            submit_btn.disabled = False
            return

        global current_job_id_tab1
        current_job_id_tab1 = job_id

        submit_btn.layout.width = '49%'
        cancel_btn_1.layout.display = 'block'
        cancel_btn_1.disabled = False
        cancel_btn_1.description = "Cancel Job"

        total_tasks = result
        progress_bar.max = total_tasks
        output_display.value = f"✅ Job {job_id} submitted!\nStarting Google Batch Job..."

        js_timer_code = """
        <script>
            // Clear any old timers just in case
            if (window.jobPollInterval) { clearInterval(window.jobPollInterval); }

            // Start a new timer that calls Python
            window.jobPollInterval = setInterval(function() {
                google.colab.kernel.invokeFunction('notebook.check_status', [], {});
            }, 15000);
        </script>
        """
        display(HTML(js_timer_code))

    def cancel_tab1_job(b):
        global current_job_id_tab1
        if current_job_id_tab1:
            cancel_btn_1.disabled = True
            cancel_btn_1.description = "Cancelling..."
            output_display.value += "\n\n🛑 Sending cancel request to GCP..."
            try:
                client = batch_v1.BatchServiceClient()
                PROJECT_ID = "paleo-interpolation"
                REGION = "australia-southeast2"
                job_name = f"projects/{PROJECT_ID}/locations/{REGION}/jobs/{current_job_id_tab1}"
                client.delete_job(name=job_name)
            except Exception as e:
                output_display.value += f"\n⚠️ Cancel failed: {str(e)}"

    cancel_btn_1.on_click(cancel_tab1_job)

    def check_dates_on_the_fly(change):
        start_val = start_date_core.value.strip()
        end_val = end_date_core.value.strip()
        error_text = ""
        dates_are_broken = False

        for val, name in [(start_val, "Start Year"), (end_val, "End Year")]:
            if val:
                if not (val.isdigit() and len(val) == 4):
                    error_text = f"<span style='color: #EE6723; font-weight: bold;'>⚠️ Error: {name} must be a 4-digit year (YYYY).</span>"
                    dates_are_broken = True
                    break

        if not dates_are_broken and start_val and end_val:
            if int(start_val) > int(end_val):
                error_text = "<span style='color: #EE6723; font-weight: bold;'>⚠️ Error: End Year must be equal to or after Start Year.</span>"
                dates_are_broken = True

        date_error_msg.value = error_text
        submit_btn.disabled = dates_are_broken

    start_date_core.observe(check_dates_on_the_fly, names='value')
    end_date_core.observe(check_dates_on_the_fly, names='value')
    submit_btn.on_click(process_data_internal)
    run_full_dataset.observe(hide_dates, names='value')

    tab_1 = widgets.VBox([
        widgets.HTML("<h3>Data Processing Interface</h3>"), spacer, checkbox_row_tab1, date_spacer, date_row, spacer,
        date_error_msg, spacer, dropdown_final, spacer, dataset_dropdown_final, spacer, dropdown_model_final, spacer, tab1_buttons, progress_bar, spacer, output_final,
    ], layout=widgets.Layout(overflow='hidden', width='100%', padding='10px'))

    # ==================================================
    # --- SHARED RUN LOADER (Tab 2 & Tab 3) ---
    # ==================================================

    def get_processing_runs():
        try:
            client = storage.Client()
            blobs = client.list_blobs('dcceew-input-data', prefix='Processing_Runs/', delimiter='/')
            list(blobs)
            if not blobs.prefixes: return [("No runs found", None)]
            options = []
            for folder_path in blobs.prefixes:
                clean_name = folder_path.split('/')[-2]
                options.append((clean_name, folder_path))
            return sorted(options, reverse=True)
        except Exception as e:
            return [(f"Error connecting to GCP", None)]

    shared_run_options = get_processing_runs()

    # ==================================================
    # --- TAB 2: DOWNLOAD & CLIP LAYOUT ---
    # ==================================================

    runs_dropdown = widgets.Dropdown(options=shared_run_options, description='Select Run:', layout=widgets.Layout(width='80%'), style=description_style)
    refresh_runs_btn = widgets.Button(description='🔄 Refresh', layout=widgets.Layout(width='18%'))
    runs_dropdown_container = widgets.HBox([runs_dropdown, refresh_runs_btn], layout=comp_layout)

    run_date_info = widgets.HTML("<b>Available Start Year:</b> -- | <b>Available End Year:</b> --", layout=widgets.Layout(margin='10px 0 10px 0'))

    def update_run_date_info(*args):
        prefix = runs_dropdown.value
        if prefix:
            clean_prefix = prefix.strip('/')
            parts = clean_prefix.split('_')
            if len(parts) >= 4:
                start_year = parts[-4]
                end_year = parts[-3]
                run_date_info.value = f"<span style='color: #EE6723; font-size: 14px;'><b>Available Start Year:</b> {start_year} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Available End Year:</b> {end_year}</span>"
            else:
                run_date_info.value = "<b>Available Start Year:</b> Unknown | <b>Available End Year:</b> Unknown"
        else:
            run_date_info.value = "<b>Available Start Year:</b> -- | <b>Available End Year:</b> --"

    runs_dropdown.observe(update_run_date_info, names='value')
    update_run_date_info()

    filter_date_cb = widgets.Checkbox(value=False, description='Filter Download by Custom Simulation Years', indent=False)

    dl_start_date = widgets.Text(placeholder='YYYY-MM-DD (e.g. 0010 or 0010-05-15)', description='Simulation Start Date', layout=comp_layout, disabled=True, style=description_style)
    dl_end_date = widgets.Text(placeholder='YYYY-MM-DD (e.g. 0010 or 0010-05-15)', description='Simulation End Date', layout=comp_layout, disabled=True, style=description_style)

    dl_date_row = widgets.HBox([
        widgets.VBox([dl_start_date], layout=date_layout_split),
        widgets.VBox([dl_end_date], layout=date_layout_split)
    ], layout=widgets.Layout(display='none', width='100%', margin='10px 0 15px 0'))

    def on_filter_date_toggle(change):
        if change['new']:
            dl_date_row.layout.display = 'flex'
            dl_start_date.disabled = False
            dl_end_date.disabled = False
        else:
            dl_date_row.layout.display = 'none'
            dl_start_date.disabled = True
            dl_end_date.disabled = True

    filter_date_cb.observe(on_filter_date_toggle, names='value')

    dl_daily_cb = widgets.Checkbox(value=True, description='Download Daily', indent=False)
    dl_monthly_cb = widgets.Checkbox(value=True, description='Download Monthly', indent=False)
    dl_yearly_cb = widgets.Checkbox(value=True, description='Download Yearly', indent=False)
    frequency_row = widgets.HBox([dl_daily_cb, dl_monthly_cb, dl_yearly_cb], layout=widgets.Layout(margin='0 0 15px 0'))

    clip_upload_cb = widgets.Checkbox(value=False, description='Upload File (.zip/.geojson)', indent=False)
    clip_draw_cb = widgets.Checkbox(value=False, description='Draw Polygon on Map', indent=False)
    checkbox_row = widgets.HBox([clip_upload_cb, clip_draw_cb], layout=widgets.Layout(margin='0 0 15px 0'))

    upload_boundary = widgets.FileUpload(
        accept='.zip,.geojson', multiple=False, description='Upload .zip (Shapefile) or .geojson',
        layout=widgets.Layout(display='none', width='100%'), style={'font_weight': 'bold'}
    )

    map_output = widgets.Output(layout=widgets.Layout(display='none', width='100%'))
    with map_output:
        display(HTML(html_map_code))

    download_btn = widgets.Button(description='Prepare & Download Data', button_style='info', layout=widgets.Layout(width='100%', height='50px'))
    download_btn.style.button_color = '#EE6723'
    download_btn.style.font_weight = 'bold'

    dl_progress = widgets.IntProgress(value=0, min=0, max=10, description='Progress:', bar_style='info', layout=widgets.Layout(overflow='hidden', width='100%', display='none'))
    dl_output = widgets.Output()

    def on_upload_toggle(change):
        if change['new']:
            clip_draw_cb.value = False
        upload_boundary.layout.display = 'flex' if clip_upload_cb.value else 'none'
        map_output.layout.display = 'block' if clip_draw_cb.value else 'none'

    def on_draw_toggle(change):
        if change['new']:
            clip_upload_cb.value = False
        upload_boundary.layout.display = 'flex' if clip_upload_cb.value else 'none'
        map_output.layout.display = 'block' if clip_draw_cb.value else 'none'

    clip_upload_cb.observe(on_upload_toggle, names='value')
    clip_draw_cb.observe(on_draw_toggle, names='value')

    def update_runs_dropdown(b):
        refresh_runs_btn.description = '⏳...'
        new_opts = get_processing_runs()
        runs_dropdown.options = new_opts
        preview_runs_dropdown.options = new_opts
        refresh_runs_btn.description = '🔄 Refresh'

    refresh_runs_btn.on_click(update_runs_dropdown)

    def process_and_download(b):
        with dl_output:
            clear_output()

            prefix_to_download = runs_dropdown.value
            if not prefix_to_download:
                print("❌ No valid run selected.")
                return

            if not prefix_to_download.endswith('/'):
                prefix_to_download += '/'

            valid_freqs = []
            if dl_daily_cb.value: valid_freqs.append('Daily')
            if dl_monthly_cb.value: valid_freqs.append('Monthly')
            if dl_yearly_cb.value: valid_freqs.append('Yearly')

            if not valid_freqs:
                print("❌ Please select at least one frequency (Daily, Monthly, or Yearly) to download.")
                return

            client = storage.Client()
            bucket = client.bucket('dcceew-input-data')
            blobs = []

            if filter_date_cb.value:
                start_filter = dl_start_date.value.strip()
                end_filter = dl_end_date.value.strip()

                if len(start_filter) >= 4 and start_filter[:4].isdigit():
                    s_year = int(start_filter[:4])
                else: s_year = 0

                if len(end_filter) >= 4 and end_filter[:4].isdigit():
                    e_year = int(end_filter[:4])
                else: e_year = 9999

                for y in range(s_year, e_year + 1):
                    decade = (y // 10) * 10
                    search_prefix = f"{prefix_to_download}{decade}/{y}/"
                    blobs.extend(list(bucket.list_blobs(prefix=search_prefix)))

                if len(start_filter) == 4 and start_filter.isdigit(): start_filter = f"{start_filter}-01-01"
                if len(end_filter) == 4 and end_filter.isdigit(): end_filter = f"{end_filter}-12-31"
            else:
                blobs = list(bucket.list_blobs(prefix=prefix_to_download))

            tif_blobs = [b for b in blobs if b.name.endswith('.tif')]

            freq_filtered_blobs = []
            for b in tif_blobs:
                if any(freq in b.name for freq in valid_freqs):
                    freq_filtered_blobs.append(b)
            tif_blobs = freq_filtered_blobs

            if not tif_blobs:
                print("❌ No files match the selected frequencies for this run.")
                return

            if filter_date_cb.value:
                date_filtered_blobs = []
                for blob in tif_blobs:
                    name = blob.name
                    file_date = None

                    if 'Daily' in name:
                        match = re.search(r'_(\d+)_(\d{2})_(\d{2})\.tif$', name)
                        if match: file_date = f"{match.group(1).zfill(4)}-{match.group(2)}-{match.group(3)}"
                    elif 'Monthly' in name:
                        match = re.search(r'_(\d+)_(\d{2})_Total\.tif$', name)
                        if match: file_date = f"{match.group(1).zfill(4)}-{match.group(2)}-01"
                    elif 'Yearly' in name:
                        match = re.search(r'_(\d+)_Total\.tif$', name)
                        if match: file_date = f"{match.group(1).zfill(4)}-01-01"

                    if file_date:
                        if start_filter <= file_date <= end_filter:
                            date_filtered_blobs.append(blob)
                    else:
                        date_filtered_blobs.append(blob)

                tif_blobs = date_filtered_blobs

                if not tif_blobs:
                    print("❌ No files match that specific custom date range.")
                    return

            dl_progress.layout.display = 'flex'
            dl_progress.value = 0
            dl_progress.max = len(tif_blobs)

            print(f"📥 1. Downloading {len(tif_blobs)} files from GCP...")

            base_dir = "/tmp/paleo_downloads"
            raw_dir = os.path.join(base_dir, "raw")
            final_dir = os.path.join(base_dir, "final")
            zip_path = "/content/Paleo_Processed_Data"

            if os.path.exists(base_dir): shutil.rmtree(base_dir)
            os.makedirs(raw_dir, exist_ok=True)
            os.makedirs(final_dir, exist_ok=True)

            downloaded_files = []

            for blob in tif_blobs:
                relative_path = blob.name.replace(prefix_to_download, "", 1)
                if relative_path.startswith('/'):
                    relative_path = relative_path[1:]

                local_file = os.path.join(raw_dir, relative_path)
                os.makedirs(os.path.dirname(local_file), exist_ok=True)

                blob.download_to_filename(local_file)
                downloaded_files.append(relative_path)
                dl_progress.value += 1

            print("✅ Download complete.")

            is_clipping = clip_upload_cb.value or clip_draw_cb.value

            if is_clipping:
                print("✂️ 2. Clipping data to custom boundary...")
                dl_progress.value = 0

                if clip_upload_cb.value:
                    if not upload_boundary.value:
                        print("❌ Clipping requested, but no file uploaded!")
                        dl_progress.layout.display = 'none'
                        return

                    uploaded_filename = list(upload_boundary.value.keys())[0]
                    file_content = upload_boundary.value[uploaded_filename]['content']
                    boundary_path = os.path.join(base_dir, uploaded_filename)
                    with open(boundary_path, "wb") as f:
                        f.write(file_content)

                    if uploaded_filename.endswith('.zip'):
                        gdf = gpd.read_file(f"zip://{boundary_path}")
                    else:
                        gdf = gpd.read_file(boundary_path)

                elif clip_draw_cb.value:
                    if global_drawn_polygon is None:
                        print("❌ Error: You checked 'Draw Polygon' but haven't drawn a shape on the map yet!")
                        dl_progress.layout.display = 'none'
                        return

                    feature_collection = {
                        "type": "FeatureCollection",
                        "features": [global_drawn_polygon]
                    }
                    boundary_path = os.path.join(base_dir, "drawn_boundary.geojson")
                    with open(boundary_path, "w") as f:
                        json.dump(feature_collection, f)

                    gdf = gpd.read_file(boundary_path)

                for rel_path in downloaded_files:
                    raw_tif = os.path.join(raw_dir, rel_path)
                    out_name = os.path.join(final_dir, rel_path)
                    os.makedirs(os.path.dirname(out_name), exist_ok=True)

                    try:
                        with rasterio.open(raw_tif) as src:
                            gdf_proj = gdf.to_crs(src.crs)
                            shapes = gdf_proj.geometry

                            out_image, out_transform = mask(src, shapes, crop=True, filled=True)
                            out_meta = src.meta.copy()

                            out_meta.update({
                                "driver": "GTiff",
                                "height": out_image.shape[1],
                                "width": out_image.shape[2],
                                "transform": out_transform,
                                "compress": "lzw",
                                "predictor": 3,
                                "tiled": True
                            })

                            with rasterio.open(out_name, "w", **out_meta) as dest:
                                dest.write(out_image)
                    except Exception as e:
                        print(f"⚠️ Failed to clip {os.path.basename(raw_tif)}: {e}")

                    dl_progress.value += 1

                print("✅ Clipping complete.")
            else:
                print("⏩ 2. Skipping clipping.")
                for rel_path in downloaded_files:
                    raw_tif = os.path.join(raw_dir, rel_path)
                    out_name = os.path.join(final_dir, rel_path)
                    os.makedirs(os.path.dirname(out_name), exist_ok=True)
                    shutil.move(raw_tif, out_name)

            print("📦 3. Zipping files for download...")
            zip_path = "/content/Paleo_Processed_Data"
            shutil.make_archive(zip_path, 'zip', final_dir)
            shutil.rmtree(base_dir)
            dl_progress.layout.display = 'none'

            print("🎉 Done! Your download will start automatically in a moment.")
            print("------------------------------------------------------------")
            print("⚠️ NOTE: Colab's automatic download can be slow for large zip files.")
            print("💡 FOR A FASTER DOWNLOAD:")
            print("   1. Click the Folder icon 📁 on the far left toolbar.")
            print("   2. Locate 'Paleo_Processed_Data.zip'.")
            print("   3. Click the three dots ⋮ next to it and select 'Download'.")
            print("------------------------------------------------------------")
            files.download(f"{zip_path}.zip")

    download_btn.on_click(process_and_download)

    tab_2 = widgets.VBox([
        widgets.HTML("<h3>Download Interface</h3>"),
        widgets.HTML("<p>Select a historical run from Google Cloud Storage.</p>"),
        runs_dropdown_container,
        run_date_info,
        spacer,
        filter_date_cb,
        dl_date_row,
        spacer,
        widgets.HTML("<b>Select Data Frequencies to Download:</b>"),
        frequency_row,
        spacer,
        widgets.HTML("<b>Select Clipping Method:</b>"),
        checkbox_row,
        upload_boundary,
        map_output,
        spacer,
        download_btn,
        dl_progress,
        widgets.HTML("<br>"),
        dl_output
    ], layout=widgets.Layout(overflow='hidden', width='100%', padding='10px'))

    # ==================================================
    # --- TAB 3: DATA PREVIEW ---
    # ==================================================

    preview_runs_dropdown = widgets.Dropdown(options=shared_run_options, description='Select Run:', layout=widgets.Layout(width='80%'), style=description_style)
    preview_refresh_btn = widgets.Button(description='🔄 Refresh', layout=widgets.Layout(width='18%'))
    preview_runs_container = widgets.HBox([preview_runs_dropdown, preview_refresh_btn], layout=comp_layout)

    def update_preview_runs_dropdown(b):
        preview_refresh_btn.description = '⏳...'
        new_opts = get_processing_runs()
        preview_runs_dropdown.options = new_opts
        runs_dropdown.options = new_opts
        preview_refresh_btn.description = '🔄 Refresh'

    preview_refresh_btn.on_click(update_preview_runs_dropdown)

    preview_run_date_info = widgets.HTML("<b>Available Start Year:</b> -- | <b>Available End Year:</b> --", layout=widgets.Layout(margin='10px 0 10px 0'))

    def update_preview_run_date_info(*args):
        prefix = preview_runs_dropdown.value
        if prefix:
            clean_prefix = prefix.strip('/')
            parts = clean_prefix.split('_')
            if len(parts) >= 4:
                start_year = parts[-4]
                end_year = parts[-3]
                preview_run_date_info.value = f"<span style='color: #EE6723; font-size: 14px;'><b>Available Start Year:</b> {start_year} &nbsp;&nbsp;|&nbsp;&nbsp; <b>Available End Year:</b> {end_year}</span>"
            else:
                preview_run_date_info.value = "<b>Available Start Year:</b> Unknown | <b>Available End Year:</b> Unknown"
        else:
            preview_run_date_info.value = "<b>Available Start Year:</b> -- | <b>Available End Year:</b> --"

    preview_runs_dropdown.observe(update_preview_run_date_info, names='value')
    update_preview_run_date_info()

    preview_freq_radio = widgets.RadioButtons(
        options=['Daily', 'Monthly', 'Yearly'],
        value='Daily',
        description='Frequency:',
        layout=comp_layout,
        style=description_style
    )

    preview_date_input = widgets.Text(
        placeholder='YYYY-MM-DD (e.g. 0010-05-15, 0010-05, or 0010)',
        description='Date to View:',
        layout=comp_layout,
        style=description_style
    )

    preview_btn = widgets.Button(description='Preview on Map', button_style='info', layout=widgets.Layout(width='100%', height='50px'))
    preview_btn.style.button_color = '#EE6723'
    preview_btn.style.font_weight = 'bold'

    preview_output = widgets.Output()

    def process_and_preview(b):
        global system_is_rendering_map

        with preview_output:
            clear_output(wait=True)

            # 🚦 Turn the light RED: Tell Tab 1 to pause its polling
            system_is_rendering_map = True

            try:
                run_prefix = preview_runs_dropdown.value
                freq = preview_freq_radio.value
                target_date = preview_date_input.value.strip()

                if not run_prefix or not target_date:
                    print("❌ Please select a run and enter a valid date.")
                    return

                if not run_prefix.endswith('/'):
                    run_prefix += '/'

                if freq == 'Monthly':
                    if len(target_date.split('-')) >= 2: target_date = "-".join(target_date.split('-')[:2])
                elif freq == 'Yearly':
                    target_date = target_date.split('-')[0]

                year_str = target_date.split('-')[0]
                if year_str.isdigit() and len(year_str) >= 1:
                    year_int = int(year_str)
                    decade_int = (year_int // 10) * 10
                    search_prefix = f"{run_prefix}{decade_int}/{year_int}/"
                else:
                    search_prefix = run_prefix

                print(f"🔍 Locating {freq} file for {target_date}...")

                client = storage.Client()
                bucket = client.bucket('dcceew-input-data')
                blobs = list(bucket.list_blobs(prefix=search_prefix))

                target_blob = None
                for blob in blobs:
                    if freq in blob.name and blob.name.endswith('.tif'):
                        name = blob.name
                        file_date = None

                        if freq == 'Daily':
                            match = re.search(r'_(\d+)_(\d{2})_(\d{2})\.tif$', name)
                            if match: file_date = f"{match.group(1).zfill(4)}-{match.group(2)}-{match.group(3)}"
                        elif freq == 'Monthly':
                            match = re.search(r'_(\d+)_(\d{2})_Total\.tif$', name)
                            if match: file_date = f"{match.group(1).zfill(4)}-{match.group(2)}"
                        elif freq == 'Yearly':
                            match = re.search(r'_(\d+)_Total\.tif$', name)
                            if match: file_date = f"{match.group(1).zfill(4)}"

                        if file_date == target_date:
                            target_blob = blob
                            break

                if not target_blob:
                    print(f"❌ No {freq} file found perfectly matching date: {target_date}")
                    return

                os.makedirs("/tmp/paleo_preview", exist_ok=True)
                local_path = "/tmp/paleo_preview/preview.tif"

                print(f"📥 Downloading: {target_blob.name.split('/')[-1]}...")
                target_blob.download_to_filename(local_path)

                print("🗺️ Rendering map...")

                with rasterio.open(local_path) as src:
                    nodata = src.nodata if src.nodata is not None else -9999.0
                    dst_crs = 'EPSG:4326'

                    # 1. Calculates mathematical transform constraints to reshape EPSG:8058 to EPSG:4326
                    transform, width, height = calculate_default_transform(
                        src.crs, dst_crs, src.width, src.height, *src.bounds
                    )

                    arr_reprojected = np.zeros((height, width), dtype=np.float32)
                    print("🗺️ Projecting data...")
                    # 2. Reprojects raster internally so Folium doesn't warp/distort the map visually
                    reproject(
                        source=rasterio.band(src, 1),
                        destination=arr_reprojected,
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=dst_crs,
                        resampling=Resampling.nearest,
                        src_nodata=nodata,
                        dst_nodata=nodata
                    )

                    # Extract properly scaled WGS84 bounds
                    bounds = transform_bounds(src.crs, dst_crs, *src.bounds)
                    folium_bounds = [[bounds[1], bounds[0]], [bounds[3], bounds[2]]]

                    arr = np.where(arr_reprojected == nodata, np.nan, arr_reprojected)
                    arr_min, arr_max = 0, np.nanmax(arr)

                    if arr_max > arr_min:
                        norm_arr = (arr - arr_min) / (arr_max - arr_min)
                    else:
                        norm_arr = np.zeros_like(arr)

                    cmap_plt = cm.get_cmap('Spectral')
                    rgba_img = cmap_plt(norm_arr)
                    rgba_img[np.isnan(arr), 3] = 0.0

                    png_path = "/tmp/paleo_preview/overlay.png"
                    plt.imsave(png_path, rgba_img)

                    center_lat = (bounds[1] + bounds[3]) / 2
                    center_lon = (bounds[0] + bounds[2]) / 2

                    f = Figure(height=400)
                    m = folium.Map(location=[center_lat, center_lon], zoom_start=5)
                    f.add_child(m)
                    print("🗺️ Finalizing Map...")
                    folium.raster_layers.ImageOverlay(
                        image=png_path,
                        bounds=folium_bounds,
                        opacity=0.8,
                        interactive=True,
                        cross_origin=False,
                        zindex=1
                    ).add_to(m)

                    if freq == 'Daily':
                        caption_mm = f"mm/day"
                    elif freq == 'Monthly':
                        caption_mm = f"mm/month"
                    elif freq == 'Yearly':
                        caption_mm = f"mm/year"

                    hex_colors = [mcolors.to_hex(cmap_plt(i)) for i in np.linspace(0, 1, 10)]
                    colormap = b_cmp.LinearColormap(colors=hex_colors, vmin=arr_min, vmax=arr_max)
                    colormap.caption = caption_mm
                    colormap.add_to(m)
                    print("🗺️ Loading Map...")

                    # --- FIX FOR BLANK MAPS IN COLAB ---
                    import html
                    map_html = f.render()

                    # Wrap the raw HTML in a srcdoc iframe to bypass Base64 URL limits
                    iframe_code = f'<iframe srcdoc="{html.escape(map_html)}" width="100%" height="420px" style="border:none;"></iframe>'
                    display(HTML(iframe_code))

            except Exception as e:
                print(f"⚠️ Failed to render map locally: {e}")

            finally:
                # 🚦 Turn the light GREEN: Tell Tab 1 it is safe to resume polling!
                system_is_rendering_map = False

    preview_btn.on_click(process_and_preview)

    tab_3 = widgets.VBox([
        widgets.HTML("<h3>Data Preview</h3>"),
        widgets.HTML("<p>Select a single GeoTIFF file from Google Cloud Storage to render an interactive map overlay.</p>"),
        preview_runs_container,
        preview_run_date_info,
        spacer,
        widgets.HTML("<b>Select Image Frequency & Date:</b>"),
        preview_freq_radio,
        preview_date_input,
        spacer,
        preview_btn,
        widgets.HTML("<br>"),
        preview_output
    ], layout=widgets.Layout(overflow='hidden', width='100%', padding='10px'))


    # ==================================================
    # --- TAB 4: Monitor progress ---
    # ==================================================

    check_btn = widgets.Button(description='Check Batch Job Status', button_style='info', layout=widgets.Layout(width='49%', height='50px'))
    check_btn.style.button_color = '#EE6723'
    check_btn.style.font_weight = 'bold'

    cancel_btn_3 = widgets.Button(description='Cancel Selected Job', button_style='danger', layout=widgets.Layout(width='49%', height='50px', margin='0 0 0 2%'))
    cancel_btn_3.style.font_weight = 'bold'
    tab4_buttons = widgets.HBox([check_btn, cancel_btn_3], layout=widgets.Layout(width='100%'))

    jobs_dropdown = widgets.Dropdown(options=[], description='Select Batch Job ID', layout=widgets.Layout(width='80%'), style=description_style)
    refresh_jobs_btn = widgets.Button(description='🔄 Refresh', layout=widgets.Layout(width='18%'))
    jobs_dropdown_container = widgets.HBox([jobs_dropdown, refresh_jobs_btn], layout=comp_layout)

    status_output = widgets.Textarea(value="", description='Job Status:', disabled=True, layout=widgets.Layout(overflow='hidden', width='100%', height='100px', display='none'), style=description_style)
    status_output_final = widgets.VBox([status_output], layout=comp_layout)

    def cancel_tab4_job(b):
        job_id = jobs_dropdown.value
        if not job_id: return

        cancel_btn_3.disabled = True
        cancel_btn_3.description = "Cancelling..."
        status_output.layout.display = 'flex'
        status_output.value = f"🛑 Sending cancel request for Job: {job_id}..."

        try:
            PROJECT_ID = "paleo-interpolation"
            REGION = "australia-southeast2"
            job_name = f"projects/{PROJECT_ID}/locations/{REGION}/jobs/{job_id}"
            client = batch_v1.BatchServiceClient()
            client.delete_job(name=job_name)
            status_output.value += "\n✅ Job successfully cancelled and deleted from Google Cloud."
            refresh_jobs_dropdown()
        except Exception as e:
            if "404" in str(e) or "Not Found" in str(e):
                status_output.value += "\n✅ Job is already deleted or could not be found."
                refresh_jobs_dropdown()
            else:
                status_output.value += f"\n❌ Error cancelling job: {str(e)}"

        cancel_btn_3.disabled = False
        cancel_btn_3.description = "Cancel Selected Job"

    cancel_btn_3.on_click(cancel_tab4_job)

    def check_single_job_status(button):
        job_id = jobs_dropdown.value
        status_output.layout.display = 'flex'

        PROJECT_ID = "paleo-interpolation"
        REGION = "australia-southeast2"
        job_name = f"projects/{PROJECT_ID}/locations/{REGION}/jobs/{job_id}"
        client = batch_v1.BatchServiceClient()

        try:
            job_data = client.get_job(name=job_name)
            state_obj = job_data.status.state
            state = state_obj.name if hasattr(state_obj, "name") else str(state_obj)
            state_map = {"1": "QUEUED", "2": "SCHEDULED", "3": "RUNNING", "4": "SUCCEEDED", "5": "FAILED"}
            state = state_map.get(state, state)

            status_task_groups = job_data.status.task_groups
            if status_task_groups and len(status_task_groups) > 0:
                group_status = list(status_task_groups.values())[0]
                counts = group_status.counts
                succeeded = counts.get("SUCCEEDED", 0)
                failed = counts.get("FAILED", 0)
                running = counts.get("RUNNING", 0)
                pending = counts.get("PENDING", 0)
                assigned = counts.get("ASSIGNED", 0)
                spec_groups = job_data.task_groups
                true_total = spec_groups[0].task_count if len(spec_groups) > 0 else 0
                active_sum = succeeded + failed + running + pending + assigned
                dormant = max(0, true_total - active_sum)

                status_output.value = (
                    f"Job ID: {job_id}\n"
                    f"Overall State: {state} | Total: {true_total}\n"
                    f"Tasks -> Succeeded: {succeeded} | Failed: {failed} | Running: {running} | Pending: {pending} | Dormant: {dormant}"
                )
            else:
                status_output.value = f"Job ID: {job_id}\nState: {state}\nAllocating infrastructure nodes..."

        except Exception as e:
            status_output.value = f"❌ Error fetching job status: {str(e)}"

    def refresh_jobs_dropdown(b=None):
        if b:
            refresh_jobs_btn.description = '⏳...'

        PROJECT_ID = "paleo-interpolation"
        REGION = "australia-southeast2"
        jobs_dropdown.options = [("Loading jobs from Google Cloud...", None)]

        try:
            client = batch_v1.BatchServiceClient()
            parent = f"projects/{PROJECT_ID}/locations/{REGION}"
            request = batch_v1.ListJobsRequest(parent=parent)
            jobs_iterable = client.list_jobs(request=request)

            raw_jobs = []
            for job in jobs_iterable:
                if job.create_time:
                    raw_jobs.append(job)

            raw_jobs.sort(key=lambda j: j.create_time, reverse=True)
            dropdown_options = []

            for job in raw_jobs:
                job_id = job.name.split('/')[-1]
                state_obj = job.status.state
                state = state_obj.name if hasattr(state_obj, "name") else str(state_obj)
                state_map = {"1": "QUEUED", "2": "SCHEDULED", "3": "RUNNING", "4": "SUCCEEDED", "5": "FAILED"}
                state = state_map.get(state, state)
                time_str = job.create_time.strftime('%Y-%m-%d %H:%M')

                display_label = f"{time_str} | {job_id} ({state})"
                dropdown_options.append((display_label, job_id))

                if len(dropdown_options) >= 50:
                    break

            if dropdown_options:
                jobs_dropdown.options = dropdown_options
            else:
                jobs_dropdown.options = [("No jobs found in this region", None)]

        except Exception as e:
            jobs_dropdown.options = [(f"Error loading jobs (See console)", None)]

        if b:
            refresh_jobs_btn.description = '🔄 Refresh'

    refresh_jobs_dropdown()
    refresh_jobs_btn.on_click(refresh_jobs_dropdown)
    check_btn.on_click(check_single_job_status)

    tab_4 = widgets.VBox([
        widgets.HTML("<h3>Progress Status</h3>"),
        jobs_dropdown_container,
        spacer,
        tab4_buttons,
        spacer,
        status_output_final
    ], layout=widgets.Layout(overflow='hidden', width='100%', padding='10px'))

    # --- FINAL TAB ASSEMBLY ---

    css_widget = widgets.HTML("""
        <style>
            .adaptive-tab-titles .lm-TabBar-tabLabel,
            .adaptive-tab-titles .p-TabBar-tabLabel {
                color: var(--jp-ui-font-color1, var(--colab-primary-text-color, inherit)) !important;
            }
        </style>
    """)

    tab_interface = widgets.Tab(children=[tab_1, tab_2, tab_3, tab_4])
    tab_interface.set_title(0, 'Processing')
    tab_interface.set_title(1, 'Download')
    tab_interface.set_title(2, 'Data Preview')
    tab_interface.set_title(3, 'Progress Status')
    tab_interface.add_class('adaptive-tab-titles')

    app_layout = widgets.VBox([tab_interface], layout=widgets.Layout(overflow='hidden', width='90%', margin='0 auto 0 auto'))

    display(css_widget)
    display(app_layout)

if __name__ == "__main__":
    interface()

HTML(value='\n        <style>\n            .adaptive-tab-titles .lm-TabBar-tabLabel,\n            .adaptive-ta…